# Fit AdEx TF to Empirical Data

This notebook fits transfer-function coefficients from a user-provided dataset (`CSV` or `NPZ`) and produces publication-ready plots.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from tvbtoolkit.brian_mf.mean_field.tf_calc import (
    TransferFunctionFitConfig,
    fit_adex_transfer_function,
    get_neuron_params_double_cell,
    save_param_set,
)

DATA_FILE = Path("./data/brian_mf/example_empirical_tf.npz")
FIG_DIR = Path("./results/figs")
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
if DATA_FILE.exists():
    data = np.load(DATA_FILE, allow_pickle=True)
    empirical = np.asarray(data["empirical"], dtype=float)
    ve = np.asarray(data["ve"], dtype=float)
    vi = np.asarray(data["vi"], dtype=float)
    adapt = np.asarray(data["adapt"], dtype=float)
else:
    # Fallback synthetic example to keep notebook runnable
    ve = np.linspace(0.1, 30.0, 30)
    vi = np.linspace(0.1, 30.0, 30)
    vve, vvi = np.meshgrid(ve, vi)
    empirical = np.clip(0.03 * vve + 0.02 * vvi + 0.1 * np.sin(vve / 5.0), 1e-4, None)
    adapt = 5e-12 + 1e-13 * vve

params = get_neuron_params_double_cell("FS-RS", si_units=False)
empirical.shape


In [ ]:
cfg = TransferFunctionFitConfig(loop_n=5, tf_maxiter=1000, vthr_maxiter=1000)
res = fit_adex_transfer_function(
    empirical,
    {
        "ve": ve,
        "vi": vi,
        "adapt": adapt,
        "params": params,
        "cell_type": "RS",
        "w_prec": False,
    },
    cfg,
)
res.rmse_hz


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

mid = empirical.shape[0] // 2
axes[0, 0].plot(ve, empirical[mid], "o", label="Empirical")
axes[0, 0].plot(ve, res.fit_rate[mid], "-", label="Fitted")
axes[0, 0].set_title("TF slice (mid inhibitory input)")
axes[0, 0].set_xlabel("ve (Hz)")
axes[0, 0].set_ylabel("f_out (Hz)")
axes[0, 0].legend(frameon=False)

residual = empirical - res.fit_rate
axes[0, 1].plot(ve, residual[mid], color="crimson")
axes[0, 1].axhline(0, color="black", lw=0.8)
axes[0, 1].set_title("Residual slice")
axes[0, 1].set_xlabel("ve (Hz)")
axes[0, 1].set_ylabel("Error (Hz)")

im0 = axes[1, 0].imshow(empirical, origin="lower", aspect="auto")
axes[1, 0].set_title("Empirical TF")
fig.colorbar(im0, ax=axes[1, 0], fraction=0.046)

im1 = axes[1, 1].imshow(res.fit_rate, origin="lower", aspect="auto")
axes[1, 1].set_title("Fitted TF")
fig.colorbar(im1, ax=axes[1, 1], fraction=0.046)

out_svg = FIG_DIR / "brain_mf_empirical_fit.svg"
out_pdf = FIG_DIR / "brain_mf_empirical_fit.pdf"
fig.savefig(out_svg, dpi=300)
fig.savefig(out_pdf, dpi=300)
(out_svg, out_pdf)


In [ ]:
meta = {
    "cell_type": "RS",
    "species": "user_defined",
    "temperature": "user_defined",
    "recording_condition": "user_defined",
    "source": "user empirical dataset",
    "toolbox_version": "0.1.0",
    "date": "2026-02-20",
}

param_file = save_param_set(
    name="empirical_rs_fit",
    params={f"P{i}": float(v) for i, v in enumerate(res.fitted_params)},
    metadata=meta,
    path="./data/brian_mf/param_db",
)
param_file
